# 📈 Notebook 03 — Đánh giá Kết quả & Trực quan hóa Heatmap

> **Mục tiêu**: Phân tích toàn diện kết quả suy diễn của mô hình CrowdDiff trên ShanghaiTech Part A.

---

## Nội dung
1. Load kết quả suy diễn từ thư mục `results/`
2. Tính **MAE** và **MSE** toàn bộ tập test
3. Bảng chi tiết sai số từng ảnh
4. **Scatter plot**: Predicted vs Ground Truth
5. **Heatmap overlay** màu JET cho từng ảnh
6. Phân tích ảnh tốt nhất / tệ nhất

In [ ]:
import sys, os, re
import numpy as np
import pandas as pd
from PIL import Image
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
print(f'📁 Thư mục kết quả: {RESULTS_DIR}')

## 1. Load Kết quả và Tính MAE / MSE

In [ ]:
# Kết quả được lưu dạng: '<tên_ảnh> <pred_count> <gt_count>.jpg'
# Ví dụ: 'IMG_1 423 389.jpg'

result_files = [f for f in os.listdir(RESULTS_DIR)
                if f.endswith('.jpg') and re.match(r'.+\d+ \d+\.jpg', f)]

records = []
for fname in result_files:
    parts = fname.replace('.jpg', '').rsplit(' ', 2)
    if len(parts) == 3:
        name, pred, gt = parts[0], int(parts[1]), int(parts[2])
        records.append({
            'name': fname,
            'pred': pred,
            'gt':   gt,
            'abs_err': abs(pred - gt),
            'sq_err':  (pred - gt) ** 2,
        })

df = pd.DataFrame(records).sort_values('abs_err').reset_index(drop=True)

if len(df) == 0:
    print('⚠️  Không tìm thấy kết quả trong thư mục results/')
    print('   Hãy chạy: bash run_pipeline.sh --steps sample trước.')
else:
    mae = df['abs_err'].mean()
    mse = np.sqrt(df['sq_err'].mean())

    print('=' * 50)
    print(f'  Tổng số ảnh test : {len(df)}')
    print(f'  MAE              : {mae:.2f}')
    print(f'  MSE (RMSE)       : {mse:.2f}')
    print('=' * 50)
    df[['name', 'pred', 'gt', 'abs_err']].head(10)

## 2. Bảng Chi tiết Sai số

In [ ]:
if len(df) > 0:
    display_df = df[['name', 'pred', 'gt', 'abs_err']].copy()
    display_df.columns = ['Tên ảnh', 'Dự đoán', 'Thực tế (GT)', 'Sai lệch tuyệt đối']
    
    # Style bảng
    styled = display_df.style \
        .background_gradient(subset=['Sai lệch tuyệt đối'], cmap='RdYlGn_r') \
        .set_caption(f'Bảng kết quả suy diễn — MAE={mae:.2f}  MSE={mse:.2f}') \
        .format({'Dự đoán': '{:.0f}', 'Thực tế (GT)': '{:.0f}', 'Sai lệch tuyệt đối': '{:.0f}'})
    display(styled)
    
    # Lưu CSV
    display_df.to_csv(os.path.join(RESULTS_DIR, 'evaluation_results.csv'), index=False)
    print('📄 Đã lưu: results/evaluation_results.csv')

## 3. Scatter Plot: Predicted vs Ground Truth

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#1a1a2e')

    # ─── Scatter Plot ───────────────────────────────────────────
    ax = axes[0]
    ax.set_facecolor('#1a1a2e')
    c = df['abs_err'].values
    sc = ax.scatter(df['gt'], df['pred'], c=c, cmap='RdYlGn_r',
                   alpha=0.8, s=80, edgecolors='white', linewidths=0.4)
    lim = max(df['gt'].max(), df['pred'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1.5, label='Lý tưởng (pred=GT)')
    ax.set_xlabel('Ground Truth (số người thực)', color='white', fontsize=11)
    ax.set_ylabel('Dự đoán (số người)',            color='white', fontsize=11)
    ax.set_title(f'Predicted vs Ground Truth\nMAE={mae:.2f}  MSE={mse:.2f}',
                 color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white')
    ax.legend(facecolor='#2a2a4e', labelcolor='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444')
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Sai số tuyệt đối', color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

    # ─── Histogram sai số ────────────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor('#1a1a2e')
    ax2.hist(df['abs_err'], bins=15, color='#5bc8f5', alpha=0.85,
             edgecolor='white', linewidth=0.5)
    ax2.axvline(mae, color='red', linestyle='--', linewidth=1.5, label=f'MAE={mae:.2f}')
    ax2.set_xlabel('Sai số tuyệt đối (pred - GT)', color='white', fontsize=11)
    ax2.set_ylabel('Số lượng ảnh', color='white', fontsize=11)
    ax2.set_title('Phân bố Sai số', color='white', fontsize=12, fontweight='bold')
    ax2.tick_params(colors='white')
    ax2.legend(facecolor='#2a2a4e', labelcolor='white')
    for spine in ax2.spines.values():
        spine.set_edgecolor('#444')

    plt.tight_layout()
    out_path = os.path.join(RESULTS_DIR, 'scatter_and_histogram.png')
    plt.savefig(out_path, dpi=120, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'📊 Đã lưu: {out_path}')

## 4. Heatmap Overlay — Top 3 Ảnh Tốt Nhất & Tệ Nhất

In [ ]:
def load_result_image(results_dir, fname):
    """Load ảnh kết quả (3 panel ghép: GT | Image | Prediction)."""
    path = os.path.join(results_dir, fname)
    return np.asarray(Image.open(path).convert('RGB'))


def show_result_panel(df_subset, title, ncols=3):
    n = len(df_subset)
    fig, axes = plt.subplots(1, n, figsize=(7*n, 4))
    if n == 1:
        axes = [axes]
    fig.patch.set_facecolor('#1a1a2e')
    fig.suptitle(title, color='white', fontsize=14, fontweight='bold', y=1.01)

    for ax, (_, row) in zip(axes, df_subset.iterrows()):
        ax.set_facecolor('#1a1a2e')
        try:
            img = load_result_image(RESULTS_DIR, row['name'])
            # Ảnh kết quả là 3 panel ghép ngang: [GT | Image | Pred]
            ax.imshow(img)
            ax.set_title(
                f"Pred={row['pred']:.0f}  GT={row['gt']:.0f}  Err={row['abs_err']:.0f}",
                color='white', fontsize=10
            )
        except Exception as e:
            ax.text(0.5, 0.5, str(e), ha='center', va='center', color='red')
        ax.axis('off')

    plt.tight_layout()
    return fig


if len(df) > 0:
    n_show = min(3, len(df))
    
    fig_best = show_result_panel(
        df.head(n_show),
        f'✅ Top {n_show} Ảnh Sai Số Thấp Nhất'
    )
    best_path = os.path.join(RESULTS_DIR, 'best_predictions.png')
    fig_best.savefig(best_path, dpi=100, bbox_inches='tight',
                     facecolor=fig_best.get_facecolor())
    plt.show()
    print(f'✅ Đã lưu: {best_path}')

    fig_worst = show_result_panel(
        df.tail(n_show),
        f'❌ Top {n_show} Ảnh Sai Số Cao Nhất'
    )
    worst_path = os.path.join(RESULTS_DIR, 'worst_predictions.png')
    fig_worst.savefig(worst_path, dpi=100, bbox_inches='tight',
                      facecolor=fig_worst.get_facecolor())
    plt.show()
    print(f'❌ Đã lưu: {worst_path}')

## 5. Heatmap Overlay Toàn bộ từ Thư mục `evaluation/heatmaps/`

In [ ]:
heatmap_dir = os.path.join(RESULTS_DIR, 'evaluation', 'heatmaps')

if os.path.isdir(heatmap_dir):
    hmap_files = sorted([f for f in os.listdir(heatmap_dir) if f.endswith('.png')])
    print(f'📁 Tìm thấy {len(hmap_files)} heatmap trong {heatmap_dir}')
    
    n_preview = min(6, len(hmap_files))
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor('#1a1a2e')
    fig.suptitle('Heatmap Overlay — Mẫu từ tập Test', color='white',
                 fontsize=15, fontweight='bold')

    for ax, fname in zip(axes.flat, hmap_files[:n_preview]):
        ax.set_facecolor('#1a1a2e')
        img = np.asarray(Image.open(os.path.join(heatmap_dir, fname)))
        ax.imshow(img)
        ax.set_title(fname[:30], color='white', fontsize=9)
        ax.axis('off')

    # Ẩn axes thừa
    for ax in axes.flat[n_preview:]:
        ax.set_visible(False)

    plt.tight_layout()
    preview_path = os.path.join(RESULTS_DIR, 'heatmap_preview.png')
    plt.savefig(preview_path, dpi=100, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f'🗺️  Preview đã lưu: {preview_path}')
else:
    print('⚠️  Chưa có heatmap. Hãy chạy evaluate.py trước:')
    print('   bash run_pipeline.sh --steps evaluate')

## 6. Tổng kết Kết quả

In [ ]:
if len(df) > 0:
    print('╔══════════════════════════════════════════════════════╗')
    print('║         KẾT QUẢ ĐÁNH GIÁ — CROWDDIFF                ║')
    print('║            Dataset: ShanghaiTech Part A               ║')
    print('╠══════════════════════════════════════════════════════╣')
    print(f'║  Số ảnh test          : {len(df):>5d}                    ║')
    print(f'║  MAE (sai số TB)      : {mae:>8.2f}                 ║')
    print(f'║  MSE (RMSE)           : {mse:>8.2f}                 ║')
    print(f'║  Sai số nhỏ nhất      : {df["abs_err"].min():>8.2f}                 ║')
    print(f'║  Sai số lớn nhất      : {df["abs_err"].max():>8.2f}                 ║')
    print(f'║  Trung vị sai số      : {df["abs_err"].median():>8.2f}                 ║')
    print('╚══════════════════════════════════════════════════════╝')
    print()
    print('📁 Các file kết quả đã xuất:')
    for fname in ['evaluation_results.csv', 'scatter_and_histogram.png',
                  'best_predictions.png', 'worst_predictions.png', 'heatmap_preview.png']:
        p = os.path.join(RESULTS_DIR, fname)
        if os.path.exists(p):
            print(f'  ✅ {p}')